# 资源分配问题

In [112]:
import numpy as np
from time import time
import random as rd

## 类定义
定义node，表示节点类

In [113]:
class node():
    def __init__(self, name=None, resourse=None):
        self.name = name
        self.resourse = resourse    # 节点所用资源的列表
        self.nets = set()     # 节点所属线网的序号列表

## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [114]:
def readNodeFile(fileName):
    file = open(fileName)
    nodes = {}
    for line in file:
        name = line.split()[0]
        resources = sum(map(int, line.split()[-10:]))
        nodes[name] = node(name, resources)
    file.close()
    return nodes

读取文件，取得线网信息

In [115]:
def readNetFile(filename):
    names = []
    netId = 0
    file = open(filename)
    for line in file:
        if 's' in line.split():
            for name in names:
                nodes[name].nets.add(netId)
            netId += 1
            names.clear()
        names.append(line.split()[0])
    nodes[name].nets.add(netId)
    file.close()

### 定义分配函数allocation

In [116]:
def allocation():
    ind = [[] for i in range(N_FPGA)]
    for node in nodes.values():
        ind[np.random.randint(N_FPGA)].append(node)
    return ind

### 定义写文件函数writeResult

In [117]:
def writeResult():
    for index, fpga in enumerate(best):
        file.write("F"+str(index)+' ')
        for node in fpga:
            file.write(node.name+' ')
        file.write('\n')

### 定义资源差异函数 resourseCal
计算各个板的资源

In [118]:
def resourseCal():
    return 0

### 定义板间连线计算函数 linkCal
计算FPGA板间连线的总和

In [119]:
def linkCal(ind):
    link = 0
    fpgaNets = [set() for i in range(N_FPGA)]
    for index, fpga in enumerate(ind):
        for node in fpga:
            fpgaNets[index] |= node.nets    # fpga网络为内部节点网络并集
    for index, net1 in enumerate(fpgaNets):
        for net2 in fpgaNets[index:]:
            link += len(net1 & net2)    # 连线数量为各板间连线之和
    return link

### 清空分配信息
清除节点中的分配信息

In [120]:
def clearAssignInfo():
    for node in nodes.values():
        node.fpga = None

## 执行代码

设定参数

In [121]:
N_FPGA = 4 # FPGA数量
MAXITER = 10**2 # 最大循环次数
PC = 0.85	# 交叉概率
PW = 0.85	# 变异概率
N = 100     # 染色体规模

读取和打开文件

In [122]:
# 读取节点文件
nodes = readNodeFile("./contest/testdata-0/design.are")

# 读取线网文件
readNetFile("./contest/testdata-0/design.net")

# 打开结果文件
file = open("./result.res", 'w')

第1种情况：

In [123]:
# clearAssignInfo()
# allocation()
# writeResult()
#file.write(str(np.std(resourseCal(nodes)))+"\n")

第2种情况：

In [124]:
# 根据染色体群体pop已经对应的适应值fitness
# 找到最高的适应值f，以及对应的染色体bst和其在pop中的编号/下标ind

def findBest():
    maxFit = max(fitness)
    index = fitness.index(maxFit)
    best = pop[index]
    return [best, maxFit, index]

In [125]:
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop

def screenPop():
    sumFit = np.cumsum(fitness)
    for ind in pop:
        randVal = np.random.rand()
        for i in range(N):
            if randVal < sumFit[i]:
                ind = pop[i]
                break

In [126]:
# 根据交叉概率PC，以及各染色体的适应值fitness，通过交叉的方式生成新群体

def crossPop():
    for i in range(int(N/2)): # 遍历雄性个体
        j = rd.randint(int(N/2),N-1)  # 寻找配偶
        if np.random.rand() < PC: # 求偶成功
            index = rd.randint(1, N_FPGA)   # 染色体交叉点
            child1 = pop[i][:index]+pop[j][index:]  # 子染色体1
            child2 = pop[j][:index]+pop[i][index:]  # 子染色体2
            pop[i], pop[j] = child1, child2  # 生成新染色体

In [127]:
def mutatePop():
    for ind in pop:
        if rd.random() < PW:    # 如果随机数小于概率就进行变异
            gene1, gene2 = rd.sample(ind, 2)
            gene1, gene2 = gene2, gene1

In [128]:
# 步骤1，产生N个染色体的初始群体,保存在pop里面

pop = [allocation() for i in range(N)] # 初始化种群

tstart = time() # 开始计时
for iter in range(MAXITER):
    links = [linkCal(ind) for ind in pop]    # 计算每个染色体的连线数量
    fitness = [1/link for link in links]  # 计算每个染色体的适应值
    best, maxFit, index = findBest()    # 找到适应度最高的个体
    screenPop() # 选取出一个新的群体
    crossPop()  # 步骤4：交叉产生新染色体，得到新群体
    mutatePop() # 步骤5：基因变异
    pop[0]=best    # 保留上一代中适应值最高的染色体
    if np.mod(iter, MAXITER/10) == 1:
        print("迭代次数： %d 运行时间： %f 秒 板间连线总数：%d" %
              (iter, time()-tstart, links[index]))

# 输出最优染色体/路径
print('最优连线数量：%f，适应值：%f，对应分配关系：\n' % (links[index], maxFit))
for index, fpga in enumerate(best):
    print("F"+str(index))
    for node in fpga:
        print(node.name,end=' ')
    print('\n')

tend = time()
print('\n问题耗时 :  %f 秒.' % (tend-tstart))

writeResult()
file.write(str(linkCal(best))+"\n")

迭代次数： 1 运行时间： 0.035550 秒 板间连线总数：129
迭代次数： 11 运行时间： 0.182567 秒 板间连线总数：86
迭代次数： 21 运行时间： 0.322437 秒 板间连线总数：86
迭代次数： 31 运行时间： 0.450319 秒 板间连线总数：83
迭代次数： 41 运行时间： 0.576508 秒 板间连线总数：72
迭代次数： 51 运行时间： 0.703413 秒 板间连线总数：72
迭代次数： 61 运行时间： 0.824993 秒 板间连线总数：72
迭代次数： 71 运行时间： 0.951174 秒 板间连线总数：71
迭代次数： 81 运行时间： 1.070215 秒 板间连线总数：65
迭代次数： 91 运行时间： 1.189234 秒 板间连线总数：65
最优连线数量：57.000000，适应值：0.017544，对应分配关系：

F0
g6 g8 g13 g21 gp6 gp18 gp21 

F1
g22 g24 gp1 gp3 gp9 gp14 gp17 

F2
g6 g8 g13 g30 gp8 gp17 

F3
g18 g26 gp4 gp8 gp12 


问题耗时 :  1.281136 秒.


3

第3种情况：

In [129]:
clearAssignInfo()
allocation()
writeResult()
file.write(str(linkCal(best))+"\n")

3

关闭文件

In [130]:
file.close()